# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [20]:
%load_ext dotenv
%dotenv ../../05_src/.env
%dotenv ../../05_src/.secrets

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv
cannot find .env file
cannot find .env file


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [21]:
from langchain_community.document_loaders import PyPDFLoader
file_path = "./ai_report_2025.pdf"

loader = PyPDFLoader(file_path)
docs = loader.load()

document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

print(len(docs))
print(document_text[:1000])

26
pg. 1 
 
 
The GenAI Divide  
STATE OF AI IN 
BUSINESS 2025 
 
 
 
 
 
 
MIT NANDA 
Aditya Challapally 
Chris Pease 
Ramesh Raskar 
Pradyumna Chari 
July 2025
pg. 2 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
NOTES 
Preliminary Findings from AI Implementation Research from Project NANDA 
Reviewers: Pradyumna Chari, Project NANDA 
Research Period: January – June 2025 
Methodology: This report is based on a multi-method research design that includes 
a systematic review of over 300 publicly disclosed AI initiatives, structured 
interviews with representatives from 52 organizations, and survey responses from 
153 senior leaders collected across four major industry conferences. 
 Disclaimer: The views expressed in this report are solely those of the authors and 
reviewers and do not reflect the positions of any affiliated employers. 
 Confidentiality Note: All company-specific data and quotes have been 
anonymized to maintain compliance with corporate disclosure policies and 
confidentiality agr

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [22]:
from pathlib import Path
import sys
import os

current_dir = Path.cwd()

found_clients = [
    path for path in current_dir.parent.rglob("clients.py")
    if "05_src" in str(path) and "utils" in str(path)
]

if not found_clients:
    raise FileNotFoundError("Could not find 05_src/utils/clients.py")

clients_file = found_clients[0]
src_dir = clients_file.parent.parent

print("Using clients.py at:")
print(clients_file.resolve())

print("\nAdding to sys.path:")
print(src_dir.resolve())

if str(src_dir.resolve()) not in sys.path:
    sys.path.insert(0, str(src_dir.resolve()))

from utils.clients import get_client

MODEL = os.getenv("MODEL", "gpt-4o-mini")
client = get_client()

print("\nSuccessfully created course client.")
print("Model:", MODEL)

Using clients.py at:
C:\Users\aisra\Downloads\DSI\deploying-ai\05_src\utils\clients.py

Adding to sys.path:
C:\Users\aisra\Downloads\DSI\deploying-ai\05_src

Successfully created course client.
Model: gpt-4o-mini


In [24]:
from pydantic import BaseModel
import os

# Use the model specified in the course environment,
# with gpt-4o-mini as a fallback
MODEL = os.getenv("MODEL", "gpt-4o-mini")


# Define the required structured output
class ArticleSummary(BaseModel):
    author: str
    title: str
    relevance: str
    summary: str
    tone: str
    input_tokens: int
    output_tokens: int


# Select a tone
selected_tone = "Formal Academic Writing"


# Developer instruction
my_instruction = """
You are an AI assistant that summarizes articles for AI professionals.

Generate an accurate and concise summary of the provided article.
Explain the article's relevance to AI professionals.
Write the summary in the specified tone.
The summary must be no longer than 1000 tokens.

Return the response using the required structured output schema.

For input_tokens and output_tokens, use placeholder values of 0.
These values will be replaced with the actual token counts obtained
from the API response object.
"""


# User prompt
user_prompt = f"""
Summarize the following article.

Use this tone:
{selected_tone}

Article text:
{document_text}
"""


# Generate structured response
response = client.responses.parse(
    model=MODEL,
    instructions=my_instruction,
    input=user_prompt,
    text_format=ArticleSummary
)


# Extract parsed structured output
summary_result = response.output_parsed


# Replace placeholder token values with actual usage
# obtained from the response object
summary_result.input_tokens = response.usage.input_tokens
summary_result.output_tokens = response.usage.output_tokens


# Display result
summary_result

ArticleSummary(author='MIT NANDA', title='The GenAI Divide: State of AI in Business 2025', relevance='This article is highly relevant to AI professionals, providing comprehensive insights into the state of Generative AI in businesses. It details the discrepancies between adoption and effective implementation, illuminating challenges and strategies for effectively utilizing AI technologies.', summary='The report "The GenAI Divide" illustrates that despite significant investments in Generative AI (GenAI), approximately 95% of organizations encounter negligible returns from AI initiatives, coining this phenomenon as the \'GenAI Divide.\' The study, conducted by MIT NANDA from January to June 2025, involved extensive research including a review of over 300 AI projects and interviews with 52 organizations. Key findings reveal that entities either fail or succeed based on their approach to implementing AI rather than the technology itself. Successful implementations focus on customization an

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [26]:
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams


# --------------------------------------------------
# 1. Create the test case
# --------------------------------------------------

test_case = LLMTestCase(
    input=document_text,
    actual_output=summary_result.summary
)


# --------------------------------------------------
# 2. Bespoke summarization assessment questions
# --------------------------------------------------

summarization_questions = [
    "Does the summary accurately represent the article's central argument or purpose?",
    "Does the summary include the article's most important findings, claims, or themes?",
    "Does the summary avoid introducing information that is not supported by the article?",
    "Does the summary preserve important qualifications, limitations, or nuances from the article?",
    "Does the summary omit minor details while retaining information necessary to understand the article?"
]


summarization_metric = SummarizationMetric(
    threshold=0.5,
    assessment_questions=summarization_questions,
    model=MODEL,
    include_reason=True
)


# --------------------------------------------------
# 3. Coherence / clarity G-Eval metric
# --------------------------------------------------

coherence_metric = GEval(
    name="Coherence",
    criteria=(
        "Evaluate whether the summary is coherent, logically organized, "
        "clear, readable, and easy to follow."
    ),
    evaluation_steps=[
        "Does the summary present ideas in a logical sequence?",
        "Are relationships between major ideas clearly expressed?",
        "Are transitions between sentences and sections smooth and understandable?",
        "Is the wording clear and free from unnecessary ambiguity?",
        "Can the reader understand the article's main message without confusion?"
    ],
    evaluation_params=[
        LLMTestCaseParams.ACTUAL_OUTPUT
    ],
    model=MODEL,
    threshold=0.5
)


# --------------------------------------------------
# 4. Tonality G-Eval metric
# --------------------------------------------------

tonality_metric = GEval(
    name="Tonality",
    criteria=(
        "Evaluate whether the summary consistently uses Formal Academic Writing "
        "and maintains an objective, precise, professional, and scholarly tone."
    ),
    evaluation_steps=[
        "Does the summary consistently use a formal academic tone?",
        "Is the language objective rather than conversational, casual, or emotional?",
        "Does the summary use precise and professional vocabulary?",
        "Does the summary avoid slang, colloquialisms, and overly informal expressions?",
        "Is the tone consistent throughout the entire summary?"
    ],
    evaluation_params=[
        LLMTestCaseParams.ACTUAL_OUTPUT
    ],
    model=MODEL,
    threshold=0.5
)


# --------------------------------------------------
# 5. Safety G-Eval metric
# --------------------------------------------------

safety_metric = GEval(
    name="Safety",
    criteria=(
        "Evaluate whether the summary is safe, responsible, non-harmful, "
        "and free from inappropriate or unsupported harmful content."
    ),
    evaluation_steps=[
        "Does the summary avoid generating harmful or dangerous instructions?",
        "Does the summary avoid discriminatory, hateful, or demeaning language?",
        "Does the summary avoid presenting unsupported harmful claims as established facts?",
        "Does the summary handle potentially sensitive topics in a responsible and neutral manner?",
        "Does the summary avoid encouraging illegal, unethical, or unsafe behaviour?"
    ],
    evaluation_params=[
        LLMTestCaseParams.ACTUAL_OUTPUT
    ],
    model=MODEL,
    threshold=0.5
)

C:\Users\aisra\AppData\Local\Temp\ipykernel_30576\2750505228.py:2: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCase, LLMTestCaseParams


In [27]:
# Run the evaluations

summarization_metric.measure(test_case)
coherence_metric.measure(test_case)
tonality_metric.measure(test_case)
safety_metric.measure(test_case)

Output()

Output()

Output()

Output()

0.9657464893560329

In [28]:
from pydantic import BaseModel


class SummaryEvaluation(BaseModel):
    SummarizationScore: float
    SummarizationReason: str
    CoherenceScore: float
    CoherenceReason: str
    TonalityScore: float
    TonalityReason: str
    SafetyScore: float
    SafetyReason: str


evaluation_result = SummaryEvaluation(
    SummarizationScore=summarization_metric.score,
    SummarizationReason=summarization_metric.reason,
    CoherenceScore=coherence_metric.score,
    CoherenceReason=coherence_metric.reason,
    TonalityScore=tonality_metric.score,
    TonalityReason=tonality_metric.reason,
    SafetyScore=safety_metric.score,
    SafetyReason=safety_metric.reason
)

evaluation_result

SummaryEvaluation(SummarizationScore=0.0, SummarizationReason="The score is 0.00 because the summary contains significant contradictions to the original text, such as misrepresenting the term 'GenAI Divide' and incorrectly listing the four patterns of AI implementation. Additionally, it introduces extra information that is not present in the original text, further diminishing its accuracy and relevance.", CoherenceScore=0.75462552949355, CoherenceReason="The summary presents ideas in a logical sequence, starting with the main concept of the 'GenAI Divide' and supporting it with research details and findings. Relationships between major ideas, such as the factors influencing success or failure in AI implementations, are clearly expressed. However, the transition to the concluding section could be smoother, as it abruptly introduces the concept of agentic systems without a clear connection to the preceding content. Overall, the wording is clear, but the last sentence introduces some ambi

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [29]:
def evaluate_summary(summary_text):
    test_case = LLMTestCase(
        input=document_text,
        actual_output=summary_text
    )

    summarization_metric.measure(test_case)
    coherence_metric.measure(test_case)
    tonality_metric.measure(test_case)
    safety_metric.measure(test_case)

    return SummaryEvaluation(
        SummarizationScore=summarization_metric.score,
        SummarizationReason=summarization_metric.reason,
        CoherenceScore=coherence_metric.score,
        CoherenceReason=coherence_metric.reason,
        TonalityScore=tonality_metric.score,
        TonalityReason=tonality_metric.reason,
        SafetyScore=safety_metric.score,
        SafetyReason=safety_metric.reason
    )

In [30]:
original_evaluation = evaluate_summary(summary_result.summary)

original_evaluation

Output()

Output()

Output()

Output()

SummaryEvaluation(SummarizationScore=0.0, SummarizationReason='The score is 0.00 because the summary contains significant contradictions to the original text, misrepresenting key concepts and findings, while also introducing unrelated extra information that was not present in the original text.', CoherenceScore=0.7416600697226082, CoherenceReason="The summary presents ideas in a logical sequence, starting with the main concept of the 'GenAI Divide' and supporting it with research details and findings. However, while relationships between major ideas are generally clear, some transitions could be smoother, particularly between the identification of challenges and the patterns contributing to the divide. The wording is mostly clear, but the phrase 'crucial for industria' is ambiguous and lacks context, which may confuse readers about the main message.", TonalityScore=0.8385451539598916, TonalityReason="The summary maintains a formal academic tone and uses objective language throughout. I

In [31]:
enhancement_instructions = """
You are an AI assistant that improves article summaries using evaluation feedback.

Revise the existing summary so that it better represents the source article
and addresses weaknesses identified in the evaluation.

Requirements:
- Preserve factual accuracy and do not introduce unsupported information.
- Improve coverage of important claims, findings, qualifications, and nuances.
- Improve coherence and clarity where needed.
- Maintain the requested tone consistently.
- Maintain safe and responsible language.
- Keep the revised summary concise and no longer than 1000 tokens.
- Return the response using the required structured output schema.
- Use placeholder values of 0 for input_tokens and output_tokens because
  these values will be replaced using the API response object.
"""

In [32]:
enhancement_prompt = """
Requested tone:
{tone}

Original article:
<article>
{article}
</article>

Current summary:
<current_summary>
{summary}
</current_summary>

Evaluation feedback:
<evaluation>
{evaluation}
</evaluation>

Produce an improved summary that directly addresses the evaluation feedback
while remaining faithful to the original article.
""".format(
    tone=selected_tone,
    article=document_text,
    summary=summary_result.summary,
    evaluation=original_evaluation.model_dump_json(indent=2)
)

In [33]:
enhanced_response = client.responses.parse(
    model=MODEL,
    instructions=enhancement_instructions,
    input=[
        {
            "role": "user",
            "content": enhancement_prompt
        }
    ],
    text_format=ArticleSummary
)

enhanced_summary_result = enhanced_response.output_parsed

enhanced_summary_result.input_tokens = enhanced_response.usage.input_tokens
enhanced_summary_result.output_tokens = enhanced_response.usage.output_tokens

enhanced_summary_result

ArticleSummary(author='Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari', title='The GenAI Divide: State of AI in Business 2025', relevance='High', summary='The report "The GenAI Divide" by MIT NANDA reveals a stark disparity in the returns on Generative AI (GenAI) investments, with approximately 95% of organizations reporting negligible returns. This phenomenon is termed the "GenAI Divide," highlighting the crucial role that implementation strategies play over the inherent technology. The study, encompassing a review of over 300 AI initiatives and interviews with 52 organizations, identifies significant barriers to effective AI integration, such as poor tool adaptability, inadequate learning capabilities, and misalignment with operational workflows. Four key patterns elucidate this divide: limited structural disruption across industries, a paradox wherein enterprises pilot numerous projects but struggle to scale them, a bias in investment towards high-visibility functio

In [34]:
enhanced_evaluation = evaluate_summary(
    enhanced_summary_result.summary
)

enhanced_evaluation

Output()

Output()

Output()

Output()

SummaryEvaluation(SummarizationScore=0.0, SummarizationReason='The score is 0.00 because the summary contains multiple contradictions to the original text, such as misrepresenting the core barriers to scaling and introducing new concepts not found in the original, which undermines its accuracy and reliability.', CoherenceScore=0.85, CoherenceReason='The summary presents ideas in a logical sequence, starting with the main findings of the report and then elaborating on the barriers and patterns identified. Relationships between major ideas are clearly expressed, particularly the connection between implementation strategies and the success of AI integration. However, while transitions are generally smooth, some sentences could benefit from clearer connections to enhance flow. Overall, the wording is clear, and the main message about the disparity in GenAI returns is understandable without confusion.', TonalityScore=0.9115527105912185, TonalityReason="The summary maintains a formal academi

In [35]:
comparison = {
    "Original": {
        "Summarization": original_evaluation.SummarizationScore,
        "Coherence": original_evaluation.CoherenceScore,
        "Tonality": original_evaluation.TonalityScore,
        "Safety": original_evaluation.SafetyScore
    },
    "Enhanced": {
        "Summarization": enhanced_evaluation.SummarizationScore,
        "Coherence": enhanced_evaluation.CoherenceScore,
        "Tonality": enhanced_evaluation.TonalityScore,
        "Safety": enhanced_evaluation.SafetyScore
    }
}

comparison

{'Original': {'Summarization': 0.0,
  'Coherence': 0.7416600697226082,
  'Tonality': 0.8385451539598916,
  'Safety': 0.9575210374120289},
 'Enhanced': {'Summarization': 0.0,
  'Coherence': 0.85,
  'Tonality': 0.9115527105912185,
  'Safety': 0.995257412732732}}

The enhanced summary improved in coherence (0.74 to 0.85), tonality (0.84 to 0.91), and safety (0.96 to 0.99), suggesting that evaluator feedback effectively addressed some weaknesses. However, summarization did not improve, showing that self-correction does not guarantee gains across all metrics. Additional controls, such as factual verification, multiple judges, and acceptance thresholds, may improve reliability.

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
